# R2b frozen hard-gated Pedestrian refinement gate

This is the final bounded experiment in the refinement family. It starts one treatment from frozen ResNet50 MonoDETR R0 epoch 185, freezes every original R0 parameter, and trains only the stride-4 projection and local box-edge refinement head for ten epochs at 1e-4. Residuals apply only when the frozen native classifier selects Pedestrian. The immutable R0 metrics are the control. A real CUDA preflight proves exact initial parity, the precise trainable set, hard gating, and finite nonzero gradients. Run top-to-bottom on a Colab GPU.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, shlex, shutil, subprocess, sys
MOBILE_REPO=Path('/content/mobile_adas3d'); MONODETR_REPO=Path('/content/MonoDETR_R2b')
MONODETR_COMMIT='6994b9f512400b258c6edb75f77423beb9c126f2'
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti'); LOCAL_DATASET_ROOT=Path('/content/kitti')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DATASET_ROOT=Path('/content/monodetr_kitti_r2b')
R0_SELECTION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json')
OFFICIAL_CHECKPOINT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/teachers/monodetr/checkpoint_best.pth')
OUTPUT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_r2b_frozen_refinement_gate')
def run(command,cwd=None,env=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    merged=os.environ.copy(); merged.update(env or {})
    result=subprocess.run(command,cwd=cwd,env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
run(['nvidia-smi'])


In [ ]:
# Fetch pinned sources and install the exact compatibility, taxonomy, logging, and matcher patches.
if not MOBILE_REPO.exists(): run(['git','clone','https://github.com/ali-rt/mobile_adas3d.git',MOBILE_REPO])
else: run(['git','pull','--ff-only'],cwd=MOBILE_REPO)
if not MONODETR_REPO.exists(): run(['git','clone','https://github.com/ZrrSkywalker/MonoDETR.git',MONODETR_REPO])
run(['git','fetch','--all'],cwd=MONODETR_REPO); run(['git','checkout',MONODETR_COMMIT],cwd=MONODETR_REPO)
run([sys.executable,'-m','pip','install','-q','gdown','pyyaml','scipy','opencv-python-headless','numba','scikit-image','tqdm','ninja','timm==1.0.20','pandas'])
for patch in ('patch_monodetr_colab_compat.py','patch_monodetr_product_taxonomy.py','patch_monodetr_verbose_resume.py','patch_monodetr_checkpoint_metadata.py','patch_monodetr_pedestrian_refinement.py','patch_monodetr_r2b_frozen_refinement.py'):
    run([sys.executable,f'scripts/{patch}','--monodetr-repo',MONODETR_REPO],cwd=MOBILE_REPO)
ops=MONODETR_REPO/'lib/models/monodetr/ops'; shutil.rmtree(ops/'build',ignore_errors=True)
run([sys.executable,'setup.py','build','install'],cwd=ops,env={'MAX_JOBS':'2'})
run([sys.executable,'-c','import torch, timm, MultiScaleDeformableAttention; from lib.models.monodetr import build_monodetr; print(torch.__version__,timm.__version__,torch.cuda.get_device_name(0))'],cwd=MONODETR_REPO)


In [ ]:
# Recreate an isolated canonical Chen-split KITTI view.
def resolve(root,names):
    for name in names:
        path=root/name
        if path.is_dir(): return path
sources={}
for key,names in {'image_2':['training/image_2','training/image_02'],'label_2':['training/label_2','training/label_02'],'calib':['training/calib']}.items():
    sources[key]=resolve(LOCAL_DATASET_ROOT,names) or resolve(DRIVE_DATASET_ROOT,names)
if any(path is None for path in sources.values()): raise FileNotFoundError(sources)
(DATASET_ROOT/'training').mkdir(parents=True,exist_ok=True); (DATASET_ROOT/'ImageSets').mkdir(parents=True,exist_ok=True)
for name,target in sources.items():
    link=DATASET_ROOT/'training'/name
    if link.is_symlink() and link.resolve()==target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target,target_is_directory=True)
for split in ('train','val'): shutil.copy2(SPLIT_DIR/f'{split}.txt',DATASET_ROOT/'ImageSets'/f'{split}.txt')
assert len((DATASET_ROOT/'ImageSets/train.txt').read_text().splitlines())==3712
assert len((DATASET_ROOT/'ImageSets/val.txt').read_text().splitlines())==3769


In [ ]:
# Rebuild canonical R0 config, freeze exact provenance, and emit the single R2b treatment config.
for required in (R0_SELECTION,OFFICIAL_CHECKPOINT):
    if not required.is_file(): raise FileNotFoundError(required)
BASE_ROOT=Path('/content/r2b_r0_base')
run([sys.executable,'scripts/prepare_monodetr_r0_reference.py','--monodetr-repo',MONODETR_REPO,'--dataset-root',DATASET_ROOT,'--official-checkpoint',OFFICIAL_CHECKPOINT,'--output-root',BASE_ROOT,'--run-name','r2b_r0_base_only'],cwd=MOBILE_REPO)
BASE_CONFIG=MONODETR_REPO/'configs/monodetr_r0_vehicle_pedestrian.yaml'
run([sys.executable,'scripts/prepare_monodetr_r2b_gate.py','--monodetr-repo',MONODETR_REPO,'--base-config',BASE_CONFIG,'--r0-selection',R0_SELECTION,'--output-root',OUTPUT_ROOT,'--gate-epochs','10','--learning-rate','1e-4','--seed','20268'],cwd=MOBILE_REPO)
MANIFEST=OUTPUT_ROOT/'r2b_gate_manifest.json'; manifest=json.loads(MANIFEST.read_text())
assert list(manifest['variants'])==['ped_refine_frozen_hard']
assert manifest['base_model_frozen'] is True and manifest['gate_mode']=='hard'
assert manifest['distillation_enabled'] is False and manifest['temperature_scaling_enabled'] is False
print(json.dumps(manifest,indent=2))


In [ ]:
# Mandatory CUDA preflight: exact R0 parity, exact trainable set, hard gate, and finite gradients.
SMOKE_REPORT=OUTPUT_ROOT/'r2b_refinement_smoke.json'
SMOKE_LOG=OUTPUT_ROOT/'colab_logs/r2b_refinement_smoke.log'; SMOKE_LOG.parent.mkdir(parents=True,exist_ok=True)
smoke_command=[sys.executable,'-u','scripts/smoke_test_monodetr_r2b_refinement.py','--monodetr-repo',str(MONODETR_REPO),'--manifest',str(MANIFEST),'--output',str(SMOKE_REPORT)]
print('+',shlex.join(smoke_command),'\nDurable log:',SMOKE_LOG,flush=True)
with SMOKE_LOG.open('w',encoding='utf-8',buffering=1) as log:
    process=subprocess.Popen(smoke_command,cwd=MOBILE_REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end='',flush=True); log.write(line)
    code=process.wait()
if code: raise RuntimeError(f'R2b smoke test exited {code}; full log: {SMOKE_LOG}')
smoke=json.loads(SMOKE_REPORT.read_text()); assert smoke['complete'] and smoke['finite_gradients']
assert smoke['gate_mode']=='hard' and smoke['base_model_frozen'] is True
assert max(smoke['initialization_max_abs_deltas'].values()) <= smoke['parity_tolerance']
print(json.dumps(smoke,indent=2))


In [ ]:
# Train only the single ten-epoch R2b treatment; completed checkpoint is skipped on rerun.
LOG_DIR=OUTPUT_ROOT/'colab_logs'; LOG_DIR.mkdir(parents=True,exist_ok=True)
name='ped_refine_frozen_hard'; variant=manifest['variants'][name]
checkpoint=Path(variant['run_dir'])/'checkpoint_epoch_10.pth'
if checkpoint.is_file(): print('cached',name,checkpoint)
else:
    log_path=LOG_DIR/f'{name}.log'; print('training',name,'log=',log_path,flush=True)
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        process=subprocess.Popen([sys.executable,'-u','tools/train_val.py','--config',variant['config']],cwd=MONODETR_REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line)
        code=process.wait()
    if code: raise RuntimeError(f'{name} exited {code}; log={log_path}')
    if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)


In [ ]:
# Complete AP + nearby recall + Pedestrian failure evaluation against immutable R0.
run([sys.executable,'-u','scripts/evaluate_monodetr_r2b_gate.py','--mobile-repo',MOBILE_REPO,'--monodetr-repo',MONODETR_REPO,'--manifest',MANIFEST,'--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR],cwd=MOBILE_REPO)
report=json.loads((OUTPUT_ROOT/'r2b_gate_comparison.json').read_text()); print(json.dumps(report,indent=2))
import pandas as pd
display(pd.read_csv(OUTPUT_ROOT/'r2b_gate_comparison.csv'))
print('Full run authorized:',report['full_run_authorized'],'selected:',report['selected'])
